# Chatomics FigRemake - Log of Terminal Commands

### **Part 1**: Downloading the '.fastq.gz' files

In [ ]:
# Create conda env 'stdbio' with specific channels (conda-forge and bioconda) and the 'fastq-dl' module
! conda create -n stdbio -c conda-forge -c bioconda fastq-dl

# Activate the conda env
! conda activate stdbio

In [ ]:
# Each fastq file shall be placed in a 'fastq' folder
! mkdir fastq

# Download the data from the Gene Expression Omnibus (GEO) using 'fastq-dl'
# 'SRR1810900' is the SRR (ID tag) for the Sequence Read Archive (SRA)
! fastq-dl --accession SRR1810900 -o fastq/ # YAP_1

# Renames the download information file to have an identifiable name
! mv fastq/fastq-run-info.tsv fastq/SRR1810900-run-info.tsv

In [ ]:
# Install the 'fastqc' module (Quality Control)
! conda install bioconda::fastqc

# Quality reports will be stored in a 'qc_reports' folder
! mkdir qc_reports

# Run the quality analysis - outputs a .HTML file to report the data in the .zip file 
! fastqc fastq/SRR1810900.fastq.gz --outdir qc_reports

### **Part 2**: Alignment

In [ ]:
# Install the 'refgenie' module for downloading genome 'assets'
! conda install refgenie

# Creates the config file for refgenie to function
# Note: init makes the /data and /alias folders and the .yaml config file
! export REFGENIE='genome_config.yaml'
! refgenie init -c $REFGENIE

# Downloads the reference human genome fasta file
! refgenie pull hg38/fasta

In [ ]:
# Install the bowtie2 module for alignments
! conda install bowtie2 -c bioconda

# Runs the alignment using 8 CPU cores to output a SAM file
# -k 1 specifies that we want the best alignment for each sequence
! bowtie2 -x GRCh38_noalt_as/GRCh38_noalt_as -U fastq/SRR1810900.fastq.gz -S fastq/SRR1810900.sam --threads 8  --no-mixed --no-discordant -k 1

In [ ]:
# Install the samtools module for handling SAM & BAM files
! conda install samtools -c bioconda

# Converts a SAM file to a BAM file
! samtools view -bS fastq/SRR1810900.sam > fastq/SRR1810900.bam
! rm fastq/SRR1810900.sam # Remove the SAM file as it's redundant

# Sorts the BAM file by chromosome and position
! samtools sort -@ 4 -o fastq/SRR1810900.sorted.bam fastq/SRR1810900.bam
! rm fastq/SRR1810900.bam # Remove the unsorted BAM file as it's redundant

# Indexes the BAM file
! samtools index fastq/SRR1810900.sorted.bam

### **Side**: Sequentially run each (due to limited storage space)
The above code is repeated (using get_bam.sh) for:
- SRR1810907 (TAZ_1)
- SRR1810912 (IgG)
- SRR1810918 (TEAD4_1)

In [ ]:
"""
Assume everything module already installed:
 - fastq-dl
 - fastqc
 - bowtie2
 - samtools
Assume folders already created (list folders here!!!!)
 - fastq
 - qc_reports
Assume hg38 (Human Genome) already downloaded in 'alias' & 'data' folders using the 'refgenie' module
Assume bowtie2 indexer already downloaded in 'GRCh38_noalt_as' folder
"""

# Gives permission to run '.sh' file
! chmod 744 get_bam.sh

# Runs the terminal commands in cycle.sh
! ./get_bam.sh

### **Part 3**: Converting to BigWig & Fetching Peaks

In [ ]:
# Install the deeptools module for converting BAM to BigWig files
! conda install -c conda-forge -c bioconda deeptools

# Convert all (but IgG, as it's the control) to BigWig files
! bamCoverage --bam fastq/SRR1810900.sorted.bam --normalizeUsing RPKM --extendReads 200 -o fastq/SRR1810900.bw
! bamCoverage --bam fastq/SRR1810907.sorted.bam --normalizeUsing RPKM --extendReads 200 -o fastq/SRR1810907.bw
! bamCoverage --bam fastq/SRR1810918.sorted.bam --normalizeUsing RPKM --extendReads 200 -o fastq/SRR1810918.bw

In [ ]:
# Install the macs3 module for determining peaks
! pip install --upgrade macs3

# Create directory to store results
! mkdir peak_reports

# Find peaks of each set, using IgG (SRR1810912) as the control
! macs3 callpeak -t fastq/SRR1810900.sorted.bam -c fastq/SRR1810912.sorted.bam -f BAM -n SRR1810900 -g hs --outdir peak_reports/SRR1810900_peak
! macs3 callpeak -t fastq/SRR1810907.sorted.bam -c fastq/SRR1810912.sorted.bam -f BAM -n SRR1810907 -g hs --outdir peak_reports/SRR1810907_peak
! macs3 callpeak -t fastq/SRR1810918.sorted.bam -c fastq/SRR1810912.sorted.bam -f BAM -n SRR1810918 -g hs --outdir peak_reports/SRR1810918_peak

# Prints the number of peaks for each set
! find . -name "*Peak"  | xargs wc -l

### **Part 4**: STILL NEED TO FINISH

Alternative in R

In [ ]:
! conda install bedtools

! bedtools intersect -a peak_reports/SRR1810907_peak/SRR1810907_peaks.narrowPeak -b peak_reports/SRR1810900_peak/SRR1810900_peaks.narrowPeak -wa | wc -l

# Unique - notice depends on which goes first (explained in README)
! bedtools intersect -a peak_reports/SRR1810907_peak/SRR1810907_peaks.narrowPeak -b peak_reports/SRR1810900_peak/SRR1810900_peaks.narrowPeak -wa | sort | uniq | wc -l

#### EXPLAIN
bedtools intersect -a peak_reports/SRR1810900_peak/SRR1810900_peaks.narrowPeak -b peak_reports/SRR1810907_peak/SRR1810907_peaks.narrowPeak -wa | sort | uniq | wc -l
